In [5]:
# =====================================================
# Walk-Forward Day-Ahead Ensemble kNN - FIXED HOUR 24
# Electricity Market - Declared Power
# =====================================================

import pandas as pd
import numpy as np
import warnings

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsRegressor

warnings.filterwarnings('ignore')

# =========================
# تنظیمات
# =========================

INPUT_FILE = "merged_output2.csv"
OUTPUT_FILE = "knn_ensemble.xlsx"

TARGET = "POWER"
DATE_COL = "DATE_MILADI"
HOUR_COL = "HOUR"
EBRAZ_COL = "ebraz"

MIN_RATIO = 1.0

NEIGHBORS_LIST = [10, 20, 40]
ENSEMBLE_WEIGHTS = [0.3, 0.4, 0.3]

# =========================
# خواندن داده
# =========================
print("خواندن داده...")
df = pd.read_csv(INPUT_FILE)
df[DATE_COL] = pd.to_datetime(df[DATE_COL])
df = df.sort_values([DATE_COL, HOUR_COL]).reset_index(drop=True)

# =========================
# ویژگی‌های زمانی پایه
# =========================
df["hour"] = df[HOUR_COL]          # ساعت واقعی 1..24
df["dayofweek"] = df[DATE_COL].dt.dayofweek
df["month"] = df[DATE_COL].dt.month
df["day"] = df[DATE_COL].dt.day
df["quarter"] = df[DATE_COL].dt.quarter

# =========================
# Lagها
# =========================
print("ایجاد Lagها...")
df["lag_24"]  = df[TARGET].shift(24)
df["lag_48"]  = df[TARGET].shift(48)
df["lag_72"]  = df[TARGET].shift(72)
df["lag_168"] = df[TARGET].shift(168)

# =========================
# ویژگی‌های پیشرفته
# =========================
print("ایجاد ویژگی‌های پیشرفته...")

df["MA_24"]  = df[TARGET].rolling(24,  min_periods=1).mean()
df["MA_168"] = df[TARGET].rolling(168, min_periods=1).mean()

df["delta_24"] = df["lag_24"] - df["lag_48"]
df["delta_48"] = df["lag_48"] - df["lag_72"]

df["lag24_hour"] = df["lag_24"] * df["hour"]
df["hour_dayofweek"] = df["hour"] * df["dayofweek"]

df["hour_sin"]  = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"]  = np.cos(2 * np.pi * df["hour"] / 24)
df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)

df["is_weekend"] = df["dayofweek"].isin([5, 6]).astype(int)
df["is_night"]   = ((df["hour"] >= 1) & (df["hour"] <= 5)).astype(int)
df["is_peak"]    = ((df["hour"] >= 17) & (df["hour"] <= 21)).astype(int)

df["ratio_24_48"]    = df["lag_24"] / (df["lag_48"] + 1)
df["ratio_24_ma168"] = df["lag_24"] / (df["MA_168"] + 1)

# =========================
# حذف NaN
# =========================
initial_rows = len(df)
df = df.dropna().reset_index(drop=True)
print(f"حذف {initial_rows - len(df)} سطر با مقادیر NaN")

# =========================
# لیست ویژگی‌ها
# =========================
FEATURES = [
    "hour", "dayofweek", "month", "quarter",
    "lag_24", "lag_48", "lag_72", "lag_168",
    "MA_24", "MA_168",
    "delta_24", "delta_48",
    "DAMA", "ROTOOBAT",
    "hour_sin", "hour_cos", "month_sin", "month_cos",
    "is_weekend", "is_night", "is_peak",
    "ratio_24_48", "ratio_24_ma168",
    "lag24_hour", "hour_dayofweek"
]

# =========================
# تنظیم هوشمند بازار
# =========================
def smart_market_adjustment(preds, context_df):
    adjusted_preds = preds.copy()
    
    lag24 = context_df["lag_24"].values
    floor = lag24 * MIN_RATIO
    
    is_peak = context_df["is_peak"].values
    peak_factor = np.where(is_peak, 0.95, 1.0)
    adjusted_preds *= peak_factor
    
    adjusted_preds = np.maximum(adjusted_preds, floor)
    
    if 'historical_max' in context_df.columns:
        ceiling = context_df['historical_max'].values * 1.25
        adjusted_preds = np.minimum(adjusted_preds, ceiling)
    
    if EBRAZ_COL in context_df.columns:
        adjusted_preds[context_df[EBRAZ_COL].values == 0] = 0
    
    adjusted_preds = np.maximum(adjusted_preds, 0)
    return adjusted_preds

# =========================
# محاسبه حداکثر تاریخی (FIXED)
# =========================
def calculate_historical_max(df, date_col, target_col):
    tmp = df.copy()
    tmp["hour_tmp"] = tmp[date_col].dt.hour + 1   # ⭐ FIX اصلی
    historical_max = tmp.groupby("hour_tmp")[target_col].max().reset_index()
    historical_max.columns = ["hour", "historical_max"]
    return historical_max

historical_max_df = calculate_historical_max(df, DATE_COL, TARGET)

# =========================
# Ensemble kNN
# =========================
class KNNEnsemble:
    def __init__(self, neighbors_list, weights):
        self.neighbors_list = neighbors_list
        self.weights = weights
        self.models = []
        
    def fit(self, X_train, y_train):
        self.models = []
        for k in self.neighbors_list:
            model = Pipeline([
                ('scaler', StandardScaler()),
                ('knn', KNeighborsRegressor(
                    n_neighbors=k,
                    weights='distance',
                    metric='minkowski',
                    p=2
                ))
            ])
            model.fit(X_train, y_train)
            self.models.append(model)
        return self
    
    def predict(self, X_test):
        predictions = []
        for model in self.models:
            predictions.append(model.predict(X_test))
        
        ensemble_pred = np.zeros_like(predictions[0])
        for pred, w in zip(predictions, self.weights):
            ensemble_pred += pred * w
        
        return ensemble_pred, predictions

# =========================
# Walk Forward Forecast
# =========================
print("\nشروع پیش‌بینی Walk-Forward ...")

df["DECLARED"] = np.nan
df["CONFIDENCE"] = np.nan
df["ENSEMBLE_VAR"] = np.nan

unique_days = sorted(df[DATE_COL].dt.date.unique())
start_day = 30

for day_idx in range(start_day, len(unique_days) - 1):

    train_days = unique_days[:day_idx]
    predict_day = unique_days[day_idx]
    
    train_idx = df[DATE_COL].dt.date.isin(train_days)
    test_idx  = df[DATE_COL].dt.date == predict_day
    
    if not test_idx.any():
        continue
    
    X_train = df.loc[train_idx, FEATURES]
    y_train = df.loc[train_idx, TARGET]
    X_test  = df.loc[test_idx, FEATURES]
    
    ensemble = KNNEnsemble(NEIGHBORS_LIST, ENSEMBLE_WEIGHTS)
    ensemble.fit(X_train, y_train)
    
    ensemble_pred, individual_preds = ensemble.predict(X_test)
    
    pred_array = np.array(individual_preds)
    ensemble_variance = np.var(pred_array, axis=0)
    
    context_df = df.loc[test_idx].copy()
    context_df = context_df.merge(historical_max_df, on="hour", how="left")
    
    adjusted_preds = smart_market_adjustment(ensemble_pred, context_df)
    
    df.loc[test_idx, "DECLARED"] = adjusted_preds
    df.loc[test_idx, "ENSEMBLE_VAR"] = ensemble_variance
    
    if ensemble_variance.max() > 0:
        df.loc[test_idx, "CONFIDENCE"] = 1 - (ensemble_variance / ensemble_variance.max())
    
    if day_idx % 30 == 0:
        print(f"پردازش روز {day_idx + 1} از {len(unique_days)}")

# =========================
# منطق نهایی
# =========================
if EBRAZ_COL in df.columns:
    df.loc[df[EBRAZ_COL] == 0, "DECLARED"] = 0

df["DECLARED"] = df["DECLARED"].clip(lower=0).round(2)

# =========================
# ذخیره خروجی
# =========================
print(f"\nذخیره نتایج در {OUTPUT_FILE}...")

err = df["DECLARED"] - df[TARGET]

results_summary = pd.DataFrame({
    'DATE_MILADI': df[DATE_COL],
    'HOUR': df[HOUR_COL],
    'Actual': df[TARGET],
    'Declared': df["DECLARED"],
    'Practical': df["practical"],
    'importance_factor': df["importance_factor"],
    'date_shamsi': df["DATE_SHAMSI"],
    'Error': err,
    'Confidence': df["CONFIDENCE"],
    'Ensemble_Var': df["ENSEMBLE_VAR"],
    'Lag_24': df["lag_24"],
    'ebraz': df[EBRAZ_COL] if EBRAZ_COL in df.columns else ''
})

with pd.ExcelWriter(OUTPUT_FILE, engine='openpyxl') as writer:
    results_summary.to_excel(writer, sheet_name='نتایج', index=False)

print("✅ پیش‌بینی تمام ساعات 1 تا 24 با موفقیت انجام شد.")
print(f"📁 فایل خروجی: {OUTPUT_FILE}")


خواندن داده...
ایجاد Lagها...
ایجاد ویژگی‌های پیشرفته...
حذف 26232 سطر با مقادیر NaN

شروع پیش‌بینی Walk-Forward ...
پردازش روز 31 از 550
پردازش روز 61 از 550
پردازش روز 91 از 550
پردازش روز 121 از 550
پردازش روز 151 از 550
پردازش روز 181 از 550
پردازش روز 211 از 550
پردازش روز 241 از 550
پردازش روز 271 از 550
پردازش روز 301 از 550
پردازش روز 331 از 550
پردازش روز 361 از 550
پردازش روز 391 از 550
پردازش روز 421 از 550
پردازش روز 451 از 550
پردازش روز 481 از 550
پردازش روز 511 از 550
پردازش روز 541 از 550

ذخیره نتایج در knn_ensemble.xlsx...
✅ پیش‌بینی تمام ساعات 1 تا 24 با موفقیت انجام شد.
📁 فایل خروجی: knn_ensemble.xlsx
